In [1]:
from langchain_milvus_ingest import store_chunks_to_milvus

C:\Users\USER\Miniconda3\envs\kpn-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from chunking import pages_to_chunks
from pdf_with_tables import extract_pdf_content

In [3]:
import os

In [4]:
from pymilvus import connections, utility, Collection

In [5]:
folder_path = 'assets'

In [6]:
# for filename in os.listdir(folder_path):
#     filepath = os.path.join(folder_path, filename)
#     print(filepath)
#     pdf_pages = extract_pdf_content(filepath)
#     filename = os.path.basename(filepath)
#     all_chunks = pages_to_chunks(pdf_pages, source=filename, language='id')
    

In [7]:
all_chunks = []
for filename in os.listdir(folder_path):
    filepath = os.path.join(folder_path, filename)
    if filepath.endswith('.pdf'):
        print(filepath)
        pdf_pages = extract_pdf_content(filepath)

        filename = os.path.basename(filepath)
        chunks = pages_to_chunks(pdf_pages, source=filename)
        all_chunks.extend(chunks)

assets\CP-Bantuan-Pengobatan-2025.pdf
assets\CP-Cuti-dan-Izin-Khusus.pdf
assets\CP_Perjalanan Dinas.pdf


In [ ]:
all_chunks[-50]

In [10]:
for chunk in all_chunks:
#     if chunk['source']=='CP_Perjalanan Dinas.pdf' and chunk['element_type']=='table_row' and chunk['page']==25:
    if chunk['source']=='CP_Perjalanan Dinas.pdf' and chunk['page']==26:
        print(chunk['chunk_text'], '\n\n')

BIAYA UANG SAKU DINAS, UANG MAKAN DINAS, DAN KELAS PENERBANGAN UNTUK PERJALANAN DINAS 1)SelainSingapore,HongKong,danJepang. 2)Dalamkondisi total jampenerbangan tersingkat ≥6(lebih dari sama dengan enam)jam, baik untukkeberangkatan ataukepulangan dalam1(satu) PDJJkeluar negeri,makaKaryawandiijinkanmenggunakankelasBusiness 


Gol.: 10.
For Gol. 10, the value for 'Kelas Penerbangan - Non Asia& Afrika, Singapore, Hong Kong, dan Jepang (dalam USD) - Makan Malam' is Business.
For Gol. 10, the value for 'Uang Makan Dinas - Asia1), Afrika (dalam USD) - Makan Malam' is Sesuai Tagihan.
For Gol. 10, the value for 'Uang Makan Dinas - Dalam Negeri (dalam ribuan Rupiah) - Makan Malam' is Sesuai Tagihan.
For Gol. 10, the value for 'Uang Makan Dinas - Dalam Negeri (dalam ribuan Rupiah) - Makan Pagi' is Sesuai Tagihan.
For Gol. 10, the value for 'Uang Makan Dinas - Dalam Negeri (dalam ribuan Rupiah) - Makan Siang' is Sesuai Tagihan.
For Gol. 10, the value for 'Uang Makan Dinas - Non Asia& Afrika, Singa

In [11]:
for chunk in all_chunks:
    if chunk['source']=='CP-Bantuan-Pengobatan-2025.pdf':
        print(chunk['chunk_text'], '\n\n')

KPN CORP BANTUAN PENGOBATAN FEB 2025 DEFINITION ● BANTUAN PENGOBATAN adalah bantuan biaya kesehatan di luar BPJS Kesehatan yang diberikan Perusahaan kepada karyawan beserta keluarganya yang sakit dan memerlukan pengobatan oleh tenaga medis atau dokter yang meliputi Bantuan Pengobatan Rawat Jalan, Bantuan Pengobatan Rawat Inap, Bantuan Biaya Persalinan, Bantuan Kacamata, dan Bantuan Pengobatan Kesehatan Khusus. 


Kacamata, dan Bantuan Pengobatan Kesehatan Khusus. 1 ● BANTUAN PENGOBATAN RAWAT JALAN adalah bantuan biaya kesehatan yang diberikan Perusahaan kepada karyawan dan keluarganya yang sakit dan memerlukan 2 pengobatan oleh tenaga medis atau dokter tetapi tidak memerlukan rawat inap. 


is atau dokter tetapi tidak memerlukan rawat inap. ● BANTUAN RAWAT INAP adalah bantuan biaya kesehatan yang diberikan Perusahaan kepada karyawan dan keluarganya yang sakit serta berdasarkan rekomendasi dan diagnosa dokter memerlukan perawatan inap di rumah sakit atau fasilitas medis lainnya yang dit

In [13]:
HOST = "127.0.0.1"
PORT = "19530"
COL = "company_policies"  # your collection name

# 1) connect
connections.connect(alias="default", host=HOST, port=PORT)

# 2) show collections
print("Collections:", utility.list_collections())

# 3) does collection exist?
print("Has collection:", utility.has_collection(COL))

# 4) inspect schema (if exists)
if utility.has_collection(COL):
    c = Collection(COL)
    print("Num entities:", c.num_entities)
    print("Schema fields:")
    for f in c.schema.fields:
        print("  ", f.name, f.dtype, getattr(f, "dim", None))

# 5) run a checksum query (example)
if utility.has_collection(COL):
    rows = c.query(expr='checksum in ["sha256-some"]', output_fields=["checksum","metadata"])
    print(rows)
coll = c

Collections: ['milvus_langchain_test', 'company_policies', 'pdf_docs_test']
Has collection: True
Num entities: 180
Schema fields:
   id 21 None
   embedding 101 768
   checksum 21 None
   metadata 23 None
data: [], extra_info: {}


In [14]:
if utility.has_collection("company_policies"):
    utility.drop_collection("company_policies")
    print("deleted")

deleted


In [15]:
skipped, inserted = store_chunks_to_milvus(
    chunks=all_chunks,
    collection_name="company_policies",
    milvus_host="127.0.0.1",
    milvus_port="19530",
    embedding_model="models/embedding-001",
    embed_batch_size=32,
    dedupe_with_milvus=True,
    dry_run=False
)
print("skipped, inserted =", skipped, inserted)


INFO:langchain_milvus_ingest:Created Milvus collection 'company_policies' with dim=768
INFO:langchain_milvus_ingest:Finished ingest: inserted=157, skipped=0


skipped, inserted = 0 157


In [ ]:
# assume coll exists from previous snippet
checksum = "sha256-your-checksum-here"
rows = coll.query(expr=f'checksum in ["{checksum}"]', output_fields=["id","checksum","metadata"])
print("Query by checksum ->", rows)
# or by chunk_id if you stored it as primary id:
chunk_id = "c-12345"
rows2 = coll.query(expr=f'id in ["{chunk_id}"]', output_fields=["id","checksum","metadata"])
print("Query by id ->", rows2)


In [16]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings  # or fallback import

In [17]:
api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
# many wrappers read env automatically; if yours needs explicit key, pass google_api_key=api_key
emb = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=api_key)

In [18]:
query_text = "Berapa biaya akomodasi untuk perjalanan dinas golongan 6 ke Sumatera?"
qvec = emb.embed_documents([query_text])[0]   # a single vector (list of floats)
print("Query vector dim:", len(qvec))

Query vector dim: 768


In [19]:
# coll from earlier
TOP_K = 5
# search params for HNSW/COSINE index — tune ef (higher -> higher recall, slower)
search_params = {"metric_type": "COSINE", "params": {"ef": 100}}

results = coll.search(
    data=[qvec],                 # list of query vectors (can search multiple at once)
    anns_field="embedding",      # the name of your vector field
    param=search_params,
    limit=TOP_K,
    output_fields=["id", "checksum", "metadata"]
)

# results is a list (one item per query) of "hits" lists
for hits in results:
    for hit in hits:
        # printing Hit attributes - depending on pymilvus version use .id or .entity.get("id")
        try:
            entity_id = hit.id
        except Exception:
            entity_id = hit.entity.get("id") if hasattr(hit, "entity") else None
        # score/distance: with COSINE higher is more similar (closer to 1). Many APIs expose .distance or .score
        score = getattr(hit, "score", None) or getattr(hit, "distance", None)
        metadata = hit.entity.get("metadata") if hasattr(hit, "entity") else None
        print("id:", entity_id, "score:", score, "checksum:", hit.entity.get("checksum") if hasattr(hit,"entity") else None, "meta:", metadata)


id: 3becef94a35246f9a18b22f0fc8ab059 score: 0.7933555841445923 checksum: 7a19f246eef1989d2fcbb3fc8028de529608f5efd6495fab6173d4cdf30655b1 meta: {'chunk_id': '3becef94a35246f9a18b22f0fc8ab059', 'chunk_text': "Gol.: 6.\nFor Gol. 6, the value for 'Kelas Penerbangan - Non Asia& Afrika, Singapore, Hong Kong, dan Jepang (dalam USD) - Makan Malam' is 70.\nFor Gol. 6, the value for 'Uang Makan Dinas - Asia1), Afrika (dalam USD) - Makan Malam' is 33.\nFor Gol. 6, the value for 'Uang Makan Dinas - Dalam Negeri (dalam ribuan Rupiah) - Makan Malam' is 50.\nFor Gol. 6, the value for 'Uang Makan Dinas - Dalam Negeri (dalam ribuan Rupiah) - Makan Pagi' is 35.\nFor Gol. 6, the value for 'Uang Makan Dinas - Dalam Negeri (dalam ribuan Rupiah) - Makan Siang' is 65.\nFor Gol. 6, the value for 'Uang Makan Dinas - Non Asia& Afrika, Singapore, Hong Kong, dan Jepang (dalam USD) - Makan Malam' is 70.\nFor Gol. 6, the value for 'Uang Saku Dinas - Asia1), Afrika (dalamUSD)' is 60.\nFor Gol. 6, the value for 'Uan

In [21]:
for hits in results:
    for hit in hits:
        # printing Hit attributes - depending on pymilvus version use .id or .entity.get("id")
        try:
            entity_id = hit.id
        except Exception:
            entity_id = hit.entity.get("id") if hasattr(hit, "entity") else None
        # score/distance: with COSINE higher is more similar (closer to 1). Many APIs expose .distance or .score
        score = getattr(hit, "score", None) or getattr(hit, "distance", None)
        metadata = hit.entity.get("metadata") if hasattr(hit, "entity") else None
#         print("id:", entity_id, "score:", score, "checksum:", hit.entity.get("checksum") if hasattr(hit,"entity") else None, "meta:", metadata)
        print(hit.get("metadata")['chunk_text'], '\n')


Gol.: 6.
For Gol. 6, the value for 'Kelas Penerbangan - Non Asia& Afrika, Singapore, Hong Kong, dan Jepang (dalam USD) - Makan Malam' is 70.
For Gol. 6, the value for 'Uang Makan Dinas - Asia1), Afrika (dalam USD) - Makan Malam' is 33.
For Gol. 6, the value for 'Uang Makan Dinas - Dalam Negeri (dalam ribuan Rupiah) - Makan Malam' is 50.
For Gol. 6, the value for 'Uang Makan Dinas - Dalam Negeri (dalam ribuan Rupiah) - Makan Pagi' is 35.
For Gol. 6, the value for 'Uang Makan Dinas - Dalam Negeri (dalam ribuan Rupiah) - Makan Siang' is 65.
For Gol. 6, the value for 'Uang Makan Dinas - Non Asia& Afrika, Singapore, Hong Kong, dan Jepang (dalam USD) - Makan Malam' is 70.
For Gol. 6, the value for 'Uang Saku Dinas - Asia1), Afrika (dalamUSD)' is 60.
For Gol. 6, the value for 'Uang Saku Dinas - Dalam Negeri (dalam ribuan Rupiah)' is 60.
For Gol. 6, the value for 'Uang Saku Dinas - Non Asia& Afrika, Singapore, Hong Kong, dan Jepang (dalam USD)' is 60. 

Gol.: 4.
For Gol. 4, the value for 'Kela

In [ ]:
# Query all chunks from page 3
rows = coll.query(
    expr="page == 3", 
    output_fields=["id", "page", "metadata"]
)
print("Results:", rows)

In [ ]:
all_chunks[-10]